<a href="https://colab.research.google.com/github/toche7/AnalyticCases/blob/main/demoMyEzpaare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import userdata

# Retrieve the API key from Colab Secrets
# Ensure you have a secret named 'MY_EZPAA_KEY' set in Colab secrets
apikey = userdata.get('MY_EZPAA_KEY')

In [3]:
import requests

url = "https://ezpaarse.cbtumu.io/"

headers = {
    "X-API-Key": apikey,
    "Log-Format-ezproxy": '%h %l %u %t "%r" %s %b',
    "Crypted-Fields": "none",
    "Output-Fields": "+id,+pii,+sudoc-ppn,-platform,-identd,-status,-size",
}

log_data = '''
1.2.3.4 - testuser [31/Jul/2026:17:01:01 +0000] "GET https://www.sciencedirect.com/science/article/pii/S0378517325001279 HTTP/1.1" 200 7544
'''.strip()

r = requests.post(
    url,
    headers=headers,
    data=log_data.encode("utf-8"),
    timeout=60
)

print(r.status_code)
print(r.text)

200
datetime;date;login;platform_name;publisher_name;rtype;mime;print_identifier;online_identifier;title_id;doi;publication_title;publication_date;unitid;domain;on_campus;log_id;ezpaarse_version;ezpaarse_date;middlewares_version;middlewares_date;platforms_version;platforms_date;middlewares;id;pii;sudoc-ppn;title;type;subject;geoip-country;geoip-latitude;geoip-longitude;host;url
2026-07-31T17:01:01+00:00;2026-07-31;testuser;Science Direct;Elsevier;ARTICLE;HTML;0378-5173;0378-5173;03785173;10.1016/j.ijpharm.2025.125291;International Journal of Pharmaceutics;2025;S0378517325001279;www.sciencedirect.com;N;88ac51fcc9ff1ef1f541c091e475b43358d0372d;3.12.0;;78f24a4;2026-09-11;bf86a56f;2026-09-23;filter, parser, deduplicator, istex, crossref, sudoc, hal, enhancer, geolocalizer, cut, on-campus-counter, qualifier, anonymizer;;S0378517325001279;;Rubicon siRNA-encapsulated liver-targeting nanoliposome is a promising therapeutic for non-alcoholic fatty liver disease;journal-article;;AU;-33.494;143.2

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
log_file_path = '/content/drive/MyDrive/EZproxyData/ezp_aug01-03_2026.log'

try:
    with open(log_file_path, 'r') as f:
        file_log_data = f.read()
    print(f"Successfully read {len(file_log_data)} characters from {log_file_path}")
except FileNotFoundError:
    print(f"Error: The file at {log_file_path} was not found. Please ensure the path is correct and your Google Drive is mounted.")
    file_log_data = None
except Exception as e:
    print(f"An error occurred while reading the file: {e}")
    file_log_data = None


Successfully read 100004554 characters from /content/drive/MyDrive/EZproxyData/ezp_aug01-03_2026.log


In [6]:
if file_log_data:
    url = "https://ezpaarse.cbtumu.io/"

    # Reusing headers from previous successful API call (cell BP71KDCGu-Fe)
    headers = {
        "X-API-Key": apikey, # 'apikey' variable is already defined in the kernel state
        "Log-Format-ezproxy": '%h %l %u %t \"%r\" %s %b',
        "Crypted-Fields": "none",
        "Output-Fields": "+id,+pii,+sudoc-ppn,-platform,-identd,-status,-size",
    }

    # Send only the first 100 lines of the log data to test with a smaller payload
    reduced_log_data = '\n'.join(file_log_data.splitlines()[:100])
    print(f"Sending a reduced log data payload ({len(reduced_log_data)} characters) to ezPAARSE API...")

    r = requests.post(
        url,
        headers=headers,
        data=reduced_log_data.encode("utf-8"),
        timeout=300 # Keep timeout high, but expect faster response for smaller data
    )

    print("Status:", r.status_code)
    print("Response:")
    # Print only the first 1000 characters of the response for brevity
    print(r.text[:1000])
else:
    print("Skipping API call because no log data was read from the file.")


Sending a reduced log data payload (19755 characters) to ezPAARSE API...
Status: 200
Response:
datetime;date;login;platform_name;publisher_name;rtype;mime;print_identifier;online_identifier;title_id;doi;publication_title;publication_date;unitid;domain;on_campus;log_id;ezpaarse_version;ezpaarse_date;middlewares_version;middlewares_date;platforms_version;platforms_date;middlewares;id;pii;sudoc-ppn;title;type;subject;geoip-country;geoip-latitude;geoip-longitude;host;url
2026-07-31T17:00:56+00:00;2026-07-31;Thaneeya.nan;Science Direct;Elsevier;SEARCH;HTML;;;;;;;;www.sciencedirect.com;N;ef177bd25618714f1859882c64993acb094c2e67;3.12.0;;78f24a4;2026-09-11;bf86a56f;2026-09-23;filter, parser, deduplicator, istex, crossref, sudoc, hal, enhancer, geolocalizer, cut, on-campus-counter, qualifier, anonymizer;;;;;;;TH;13.7512;100.5172;49.49.235.38;https://www.sciencedirect.com:443/search?qs=Rubicon%20siRNA-encapsulated%20liver-targeting%20nanoliposome%20is%20a%20promising%20therapeutic%20for%20non-al

In [ ]:
import os
import io
import time
import requests
import pandas as pd

# =========================
# CONFIG
# =========================

INPUT_FILE = '/content/drive/MyDrive/EZproxyData/ezp_aug01-03_2026.log'
OUTPUT_FILE = "ezpaarse_output.csv"

URL = "https://ezpaarse.cbtumu.io/"
API_KEY = apikey

CHUNK_SIZE = 30000          # lines per request
TIMEOUT = 600               # seconds
MAX_RETRIES = 3
RETRY_WAIT = 5              # seconds

headers = {
    "X-API-Key": API_KEY,
    "Log-Format-ezproxy": '%h %l %u %t "%r" %s %b',
    "Crypted-Fields": "none",
    "Output-Fields": "+id,+pii,+sudoc-ppn,-platform,-identd,-status,-size",
}

expected_columns = [
    "datetime", "date", "login", "platform_name", "publisher_name",
    "rtype", "mime", "print_identifier", "online_identifier",
    "title_id", "doi", "publication_title", "publication_date",
    "unitid", "domain", "on_campus", "log_id", "ezpaarse_version",
    "ezpaarse_date", "middlewares_version", "middlewares_date",
    "platforms_version", "platforms_date", "middlewares",
    "id", "pii", "sudoc-ppn", "title", "type", "subject",
    "geoip-country", "geoip-latitude", "geoip-longitude",
    "host", "url"
]

# =========================
# HELPERS
# =========================

def send_chunk(chunk_lines, chunk_no):
    """
    Send one chunk to ezPAARSE.
    Returns DataFrame or None if failed.
    """

    payload = "".join(chunk_lines)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(
                f"Chunk {chunk_no} | "
                f"{len(chunk_lines):,} lines | "
                f"{len(payload) / 1024 / 1024:.2f} MB | "
                f"attempt {attempt}/{MAX_RETRIES}"
            )

            r = requests.post(
                URL,
                headers=headers,
                data=payload.encode("utf-8"),
                timeout=TIMEOUT
            )

            if r.status_code != 200:
                print(
                    f"HTTP {r.status_code}: "
                    f"{r.text[:300].replace(chr(10), ' ')}"
                )

            else:
                if not r.text.strip():
                    print("Empty response")
                    return pd.DataFrame(columns=expected_columns)

                try:
                    df = pd.read_csv(
                        io.StringIO(r.text),
                        sep=";",
                        dtype=str,
                        keep_default_na=False
                    )

                    print(f"  -> {len(df):,} output rows")
                    return df

                except pd.errors.EmptyDataError:
                    print("Response contained no CSV rows")
                    return pd.DataFrame(columns=expected_columns)

                except Exception as e:
                    print(f"CSV parse error: {e}")
                    print(r.text[:500])

            if attempt < MAX_RETRIES:
                print(f"Retrying in {RETRY_WAIT}s...")
                time.sleep(RETRY_WAIT)

        except requests.exceptions.Timeout:
            print(f"Timeout after {TIMEOUT}s")

            if attempt < MAX_RETRIES:
                time.sleep(RETRY_WAIT)

        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")

            if attempt < MAX_RETRIES:
                time.sleep(RETRY_WAIT)

    return None


def append_dataframe(df, output_file):
    """
    Append chunk result to CSV without keeping all data in RAM.
    """

    if df is None or df.empty:
        return

    # Keep canonical column order but preserve new columns too
    canonical = [c for c in expected_columns if c in df.columns]
    extra = [c for c in df.columns if c not in expected_columns]

    df = df[canonical + extra]

    file_exists = os.path.exists(output_file)

    df.to_csv(
        output_file,
        mode="a",
        index=False,
        header=not file_exists,
        encoding="utf-8"
    )


# =========================
# MAIN PROCESS
# =========================

failed_chunks = []

chunk_lines = []
chunk_no = 0
total_input_lines = 0
total_output_rows = 0

start_time = time.time()

print(f"Input : {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print(f"Chunk : {CHUNK_SIZE:,} lines")
print()

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for line in f:

        chunk_lines.append(line)
        total_input_lines += 1

        if len(chunk_lines) >= CHUNK_SIZE:

            chunk_no += 1

            df = send_chunk(
                chunk_lines,
                chunk_no
            )

            if df is None:
                failed_chunks.append(chunk_no)
            else:
                append_dataframe(
                    df,
                    OUTPUT_FILE
                )
                total_output_rows += len(df)

            chunk_lines = []

            elapsed = time.time() - start_time

            print(
                f"Processed input: {total_input_lines:,} lines | "
                f"Output: {total_output_rows:,} rows | "
                f"Elapsed: {elapsed / 60:.1f} min"
            )
            print("-" * 70)


# Final partial chunk
if chunk_lines:

    chunk_no += 1

    df = send_chunk(
        chunk_lines,
        chunk_no
    )

    if df is None:
        failed_chunks.append(chunk_no)
    else:
        append_dataframe(
            df,
            OUTPUT_FILE
        )
        total_output_rows += len(df)


# =========================
# SUMMARY
# =========================

elapsed = time.time() - start_time

print()
print("=" * 70)
print("FINISHED")
print("=" * 70)

print(f"Input lines : {total_input_lines:,}")
print(f"Output rows : {total_output_rows:,}")
print(f"Chunks      : {chunk_no:,}")
print(f"Time        : {elapsed / 60:.2f} minutes")

if failed_chunks:
    print(f"Failed chunks: {failed_chunks}")
else:
    print("Failed chunks: none")

print(f"Output file : {OUTPUT_FILE}")

Input : /content/drive/MyDrive/EZproxyData/ezp_aug01-03_2026.log
Output: ezpaarse_output.csv
Chunk : 30,000 lines

Chunk 1 | 30,000 lines | 6.13 MB | attempt 1/3
  -> 507 output rows
Processed input: 30,000 lines | Output: 507 rows | Elapsed: 0.8 min
----------------------------------------------------------------------
Chunk 2 | 30,000 lines | 6.57 MB | attempt 1/3
  -> 604 output rows
Processed input: 60,000 lines | Output: 1,111 rows | Elapsed: 1.8 min
----------------------------------------------------------------------
Chunk 3 | 30,000 lines | 5.72 MB | attempt 1/3
  -> 555 output rows
Processed input: 90,000 lines | Output: 1,666 rows | Elapsed: 2.1 min
----------------------------------------------------------------------
Chunk 4 | 30,000 lines | 5.09 MB | attempt 1/3
  -> 505 output rows
Processed input: 120,000 lines | Output: 2,171 rows | Elapsed: 2.7 min
----------------------------------------------------------------------
Chunk 5 | 30,000 lines | 5.80 MB | attempt 1/3
  -